In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
# user-defined paths
base_dir = '../../../../datos-notebooks/3.global-data-cba/'
ground_data_name = 'ground_data_160226_t1_t2.csv'

In [ ]:
# derived paths
ground_data_path = os.path.join(base_dir, 'MRT-prediction/ground_data', ground_data_name)

#### computing differences

TA1, TG1: Air temperature and Globe temperature - thermometer 1 (Sper Scientific 800036)

TA2, TG2: Air temperature and Globe temperature - thermometer 2 (TM-188D TENMARS )

ANEMOMETRO: wind speed in km/h with cup anemometer

CONDICION CIELO: sky condition

In [ ]:
df = pd.read_csv(ground_data_path, sep=";")
df

In [ ]:
# convert to numeric

cols = ["TG1", "TG2", "TA1", "TA2", "ANEMOMETRO"]

df[cols] = (
    df[cols]
    .astype(str)
    .apply(lambda x: x.str.replace(",", ".", regex=False))
    .apply(pd.to_numeric, errors="coerce")
)

# TG difference
df["dTG"] = df["TG1"] - df["TG2"]

In [ ]:
# Scatter: TG1 vs dTG
plt.scatter(df["TG1"], df["dTG"])
plt.xlabel("TG1")
plt.ylabel("Difference in TG")
plt.show()

In [ ]:
# fitting linear regression to correct TG1 based on TG2

X = df[["TG1"]]
y = df["TG2"]

# Fit 
model = LinearRegression()
model.fit(X, y)

# Coefficients
a = model.coef_[0]
b = model.intercept_

print("Applied correction:")
print(f"TG1_corrected = {a:.3f} * TG1 + {b:.3f}")

# Apply correction: "pass" TG1 to TG2-equivalent
df["TG1_corr"] = a * df["TG1"] + b
df["dTG_CORR"] = df["TG1_corr"] - df["TG2"]

reduction_rmse = (
    1 - np.sqrt((df["dTG_CORR"]**2).mean()) /
        np.sqrt((df["dTG"]**2).mean())
) * 100

print(f"RMSE reduction: {reduction_rmse:.1f} %")


In [ ]:
# Scatter: TG1 vs dTG
plt.scatter(df["TG1"], df["dTG_CORR"])
plt.xlabel("TG1")
plt.ylabel("Difference in TG with TG1 adjusted")
plt.show()

In [ ]:
metrics = pd.DataFrame({
    "metric": ["mean", "std", "RMSE", "MAE"],
    "dTG_raw": [
        df["dTG"].mean(),
        df["dTG"].std(),
        np.sqrt((df["dTG"]**2).mean()),
        df["dTG"].abs().mean()
    ],
    "dTG_corr": [
        df["dTG_CORR"].mean(),
        df["dTG_CORR"].std(),
        np.sqrt((df["dTG_CORR"]**2).mean()),
        df["dTG_CORR"].abs().mean()
    ]
})

print(metrics)


In [ ]:
plt.boxplot(
    [df["dTG"].dropna(), df["dTG_CORR"].dropna()],
    labels=["dTG raw", "dTG corrected"]
)
plt.ylabel("TG1 − TG2")
plt.show()
